# Post 3 — Feature build walkthrough (§4.3, §4.4)

Documents the five nested feature levels (`minimal -> lag -> rolling -> promo -> full`), shows the column set growing monotonically, and demonstrates the `as_of` leakage guard on one series. Ends by running `tests/test_leakage.py` so the notebook doubles as a feature-pipeline gate. This is model-agnostic — no TabPFN dependency.

## 0. Control panel

In [1]:
# ---- Control panel (feature build §4.3) -----------------------------------
CONFIG_NAME = "default.yaml"
OUTPUT_SUBDIR = "feature_build"
USE_CORE_SLICE = True
EXAMPLE_AS_OF = "2017-06-11"      # a test origin; lags/rollups freeze at <= this date
QUICK = True
RANDOM_SEED = 42

## 1. Imports

In [2]:
import sys, time, json, copy, platform, subprocess
from pathlib import Path

# Make `src` importable whether the notebook runs from notebooks/ or repo root.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "demand").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.demand.config import load_config, resolve_path
from src.demand.data.load import load_raw_data, build_panel
from src.demand.data.splits import (
    build_main_splits, build_core_slice, family_regime, regime_horizon, regime_costs,
)
from src.demand.data.supervised_frame import build_supervised_frame
from src.demand.models import lgbm_point, lgbm_quantile
from src.demand.models.lgbm_base import train_lgbm

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "no-git"

## 2. Load data + scope

In [3]:
cfg = copy.deepcopy(load_config(CONFIG_NAME))
cfg["seed"] = RANDOM_SEED
splits = build_main_splits(cfg)

t0 = time.time()
raw = load_raw_data(cfg)
panel = build_panel(raw, include_test=False)
print(f"loaded panel in {time.time()-t0:.1f}s | rows={len(panel):,} | "
      f"series={panel[['store_nbr','family']].drop_duplicates().shape[0]:,}")

if USE_CORE_SLICE:
    series_filter = build_core_slice(panel, raw.stores, cfg, write_csv=False)
    print(f"core slice: {len(series_filter)} series, "
          f"{series_filter['family'].nunique()} families")
else:
    series_filter = None
    print("scope: full panel")

results_dir = resolve_path(cfg, "results_dir")
figures_dir = resolve_path(cfg, "figures_dir")
out_dir = results_dir / OUTPUT_SUBDIR
out_dir.mkdir(parents=True, exist_ok=True)
print(f"git commit: {git_commit()}  ->  output: {out_dir}")

/Users/alexruppelt/Documents/Projects/tabpfn_analysis/src/demand/data/load.py:88: UserWarning: train.csv panel not complete: expected 3008016 rows, got 3000888; gaps will be zero-filled in build_panel().
  warnings.warn(


loaded panel in 3.8s | rows=3,008,016 | series=1,782
core slice: 100 series, 33 families
git commit: c58e1c9  ->  output: /Users/alexruppelt/Documents/Projects/tabpfn_analysis/results/feature_build


/Users/alexruppelt/Documents/Projects/tabpfn_analysis/src/demand/data/splits.py:205: UserWarning: core-slice: used best-available sparse-family fallback for families ['BOOKS']
  warnings.warn(


## 3. The five nested feature levels

In [4]:
from src.demand.data.features import (
    FEATURE_SETS, build_features, feature_columns, CATEGORICAL_COLS,
)

as_of = pd.Timestamp(EXAMPLE_AS_OF)

# The five nested levels grow monotonically; show how the column set expands.
rows = []
prev = set()
for level in FEATURE_SETS:
    cols = set(feature_columns(level, include_earthquake=True))
    rows.append({"feature_set": level, "n_features": len(cols),
                 "added": ", ".join(sorted(cols - prev)) or "(base)"})
    prev = cols
display(pd.DataFrame(rows))

,feature_set,n_features,added
0,minimal,13,"city, cluster, day_of_month, day_of_week, fami..."
1,lag,17,"sales_lag_1, sales_lag_14, sales_lag_28, sales..."
2,rolling,21,"sales_ma_28, sales_ma_7, sales_std_28, sales_s..."
3,promo,27,"days_to_nearest_holiday, days_to_nearest_payda..."
4,full,34,"days_since_earthquake, earthquake_flag, family..."


## 4. Leakage guard (`as_of`) on one series

In [5]:
# §4.4 leakage demonstration: build at `as_of` and confirm no lag/rolling column
# uses any sales dated after the origin. (The unit tests in tests/test_leakage.py
# assert this formally; here we eyeball one series.)
sample_pair = series_filter.iloc[0] if series_filter is not None else     panel[["store_nbr", "family"]].drop_duplicates().iloc[0]
store, fam = int(sample_pair["store_nbr"]), sample_pair["family"]
one = panel[(panel["store_nbr"] == store) & (panel["family"] == fam)]

feats_full = build_features(panel, feature_set="full", as_of=as_of, config=cfg,
                            holidays=raw.holidays, stores=raw.stores)
fp = feats_full[(feats_full["store_nbr"] == store) & (feats_full["family"] == fam)]
hist_cols = [c for c in ["sales", "sales_lag_1", "sales_lag_7", "sales_ma_7",
                          "sales_ma_28", "onpromotion", "oil_price"] if c in fp.columns]
print(f"series {store}__{fam} around as_of={as_of.date()}:")
display(fp[(fp["date"] >= as_of - pd.Timedelta(days=3)) &
          (fp["date"] <= as_of + pd.Timedelta(days=3))][["date", *hist_cols]])

series 35__AUTOMOTIVE around as_of=2017-06-11:


,date,sales,sales_lag_1,sales_lag_7,sales_ma_7,sales_ma_28,onpromotion,oil_price
1895555,2017-06-08,3.0,9.0,4.0,6.857143,5.928571,0,45.68
1895556,2017-06-09,19.0,3.0,7.0,6.714286,5.678571,0,45.82
1895557,2017-06-10,13.0,19.0,3.0,8.428572,6.178571,0,45.82
1895558,2017-06-11,3.0,13.0,10.0,9.857142,6.535714,0,45.82
1895559,2017-06-12,0.0,3.0,2.0,8.857142,6.428571,0,45.82
1895560,2017-06-13,6.0,NaN,13.0,10.000000,6.592593,0,45.82
1895561,2017-06-14,4.0,NaN,9.0,9.400000,6.769231,0,45.82


## 5. Formal leakage tests

In [6]:
# Run the formal leakage unit tests so this notebook doubles as a gate (§4.4).
import subprocess
res = subprocess.run(["python", "-m", "pytest", "tests/test_leakage.py", "-q"],
                     cwd=str(REPO_ROOT), capture_output=True, text=True)
print(res.stdout[-3000:])
print(res.stderr[-1500:])

............                                                             [100%]
12 passed in 0.77s


